# ДЗ9 · Этап 5 · Пост-хок прогон U1 со схемой ДЗ8
Выберите Colab T4. Положите bundle_posthoc.zip в MyDrive/HW9_Ranker/posthoc. Запускайте ячейки сверху вниз. Отдельные журналы сохраняются на Drive. При возобновлении повторно запускайте ячейки с начала в том же окружении. Июньские CSV и метки этому прогону не нужны.

In [ ]:
# 1. Drive and paths.
from pathlib import Path
import os
import sys

DRY_RUN = False
if DRY_RUN:
    workspace = Path(os.environ["HW9_POSTHOC_DRY_WORKSPACE"])
    bundle = workspace / "bundle"
    results = workspace / "dry_results"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    workspace = Path("/content/drive/MyDrive/HW9_Ranker/posthoc")
    bundle = Path("/content/hw9_posthoc_bundle")
    results = workspace / "posthoc_results"
archive = workspace / "bundle_posthoc.zip"
results.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Verify the frozen bundle and each prompt before importing runtime code.
import hashlib
import json
from zipfile import ZipFile

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

expected_archive = "fcd96954ae85ae2b53bec2bff92e9b5750751a3425e1a2c760c1cfeea6a25ee2"
assert file_hash(archive) == expected_archive, "Wrong bundle_posthoc.zip"
with ZipFile(archive) as zipped:
    manifest = json.loads(zipped.read("manifest.json"))
    members = manifest["file_sha256"]
    assert sorted(zipped.namelist()) == sorted([*members, "manifest.json"])
    assert all(Path(name).name == name and name not in {".", ".."} for name in members)
    for name, checksum in members.items():
        assert hashlib.sha256(zipped.read(name)).hexdigest() == checksum, name
    zipped.extractall(bundle)

sys.path.insert(0, str(bundle))
from stage3_runtime import append, check_identity, read_jsonl, run_set
sets = ("main_g", "ablation_g_seed0", "ablation_g_seed1", "ablation_g_seed2")
records = {name: read_jsonl(bundle / f"{name}.jsonl") for name in sets}
assert manifest["prompt_counts"] == {name: 100 for name in sets}
for name, rows in records.items():
    assert [r["id"] for r in rows] == [f"{name}:U{i:03d}" for i in range(1, 101)]
    for row in rows:
        assert hashlib.sha256(row["prompt"].encode()).hexdigest() == row["prompt_sha256"] == manifest["prompt_sha256"][row["id"]]
gen = manifest["generation"]
print("Verified:", sum(map(len, records.values())), "frozen prompts")

In [ ]:
# 3. Match the HW8/HW9 dependencies and verify the pinned checkpoint.
if not DRY_RUN:
    import subprocess
    import time
    hardware = subprocess.run(["nvidia-smi"], check=True, capture_output=True, text=True).stdout
    print(hardware)
    (results / "nvidia-smi.log").write_text(hardware, encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6",
                    "huggingface_hub==0.36.0", "hf_xet==1.2.0", "psutil==7.2.2"], check=True)
    import torch
    import transformers
    from huggingface_hub import snapshot_download

    assert torch.cuda.is_available()
    gpu = torch.cuda.get_device_name()
    assert "T4" in gpu, f"Select Colab T4, current GPU: {gpu}"
    started = time.perf_counter()
    weights = Path(snapshot_download(repo_id=gen["model"], revision=gen["revision"],
                                     allow_patterns=list(manifest["model_files_sha256"]), token=False))
    assert weights.name == gen["revision"]
    weight_hashes = {name: file_hash(weights / name) for name in manifest["model_files_sha256"]}
    assert weight_hashes == manifest["model_files_sha256"]
    identity = {"bundle_sha256": expected_archive, "revision": gen["revision"], "gpu": gpu,
                "torch": torch.__version__, "transformers": transformers.__version__,
                "generation": gen, "weight_sha256": weight_hashes}
    check_identity(results / "run_identity.json", identity)
    append(results / "timing.jsonl", {"stage": "checkpoint_and_verification",
                                     "seconds": time.perf_counter() - started})
    print({"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
           "revision": gen["revision"]})

In [ ]:
# 4. Generate raw responses in frozen order and resume from per-set journals.
if DRY_RUN:
    class Fake:
        calls = 0
        def generate(self, prompt):
            self.calls += 1
            return {"text": "synthetic response", "input_tokens": len(prompt.split()),
                    "output_tokens": 2, "seconds": 0.0}

    trial = records["main_g"][:2]
    path = results / "main_g.jsonl"
    backend = Fake()
    first = run_set(trial[:1], path, backend)
    resumed = run_set(trial, path, backend)
    reopened = run_set(trial, path, backend)
    assert [(x["skipped"], x["new"]) for x in (first, resumed, reopened)] == [(0, 1), (1, 1), (2, 0)]
    assert backend.calls == 2 and [r["id"] for r in read_jsonl(path)] == [r["id"] for r in trial]
    print("Dry-run:", backend.calls, "synthetic calls; resumed:", resumed, "reopened:", reopened)
else:
    from gpu_llm import GPUQwen
    import traceback
    lengths = {name: len(read_jsonl(results / f"{name}.jsonl")) for name in sets}
    try:
        backend = None
        if any(lengths[name] < len(records[name]) for name in sets):
            started = time.perf_counter()
            backend = GPUQwen(weights, gen)
            append(results / "timing.jsonl", {"stage": "model_load", "dtype": backend.metadata["dtype"],
                                             "seconds": time.perf_counter() - started})
        for name in sets:
            journal = results / f"{name}.jsonl"
            completed = lengths[name]
            append(results / "timing.jsonl", {"stage": name, "event": "start", "completed": completed})
            try:
                summary = run_set(records[name], journal, backend)
            finally:
                saved = read_jsonl(journal)
                append(results / "timing.jsonl", {"stage": name, "event": "end",
                                                 "completed": len(saved), "new": len(saved) - completed,
                                                 "new_generation_seconds": sum(row["seconds"] for row in saved[completed:])})
            print(name, summary)
    except Exception:
        with (results / "errors.log").open("a", encoding="utf-8") as log:
            traceback.print_exc(file=log)
        raise

In [ ]:
# 5. Download the raw journals and runtime logs, including after interruption.
if not DRY_RUN:
    from google.colab import files as colab_files
    from zipfile import ZIP_DEFLATED
    download = workspace / "hw9_posthoc_colab_results.zip"
    with ZipFile(download, "w", ZIP_DEFLATED) as zipped:
        for source in sorted(results.iterdir()):
            if source.is_file() and source.suffix in {".json", ".jsonl", ".log"}:
                zipped.write(source, source.name)
    colab_files.download(str(download))